# Milestone 20 - Performance Integration

This notebook traces the integration of the standalone Performance Testing Agent into the main LangGraph workflow.

## Objective

Performance Testing is integrated only after standalone validation. The agent remains safe by default: small load, GET/HEAD only, and localhost-only unless explicitly allowed.

## Integrated Workflow

```text
START -> orchestrator -> repo_analyzer -> rag -> test_planner -> api_testing -> ui_testing -> performance_testing -> bug_analysis -> report -> END
```

## State Handoff

- Test Planner writes `performance_tests` or API tests that can be used to infer safe performance checks.
- Performance Agent reads `target_url`, `test_plan`, `user_preferences`, and `discovered_endpoints`.
- Performance Agent writes `performance_results`, `performance_result_path`, and `performance_artifacts`.
- Bug Analysis reads performance results and classifies anomalies.
- Report displays performance results and summary metrics.

Target repository: https://github.com/Vitaee/DjangoRestAPI

Target URL: http://localhost:8000

## Part A - Fake repo integration with mocked execution

In [ ]:
from pathlib import Path
from test_auto.agents import api_testing_agent, ui_testing_agent, performance_testing_agent
from test_auto.graph.workflow import run_workflow

repo = Path("notebook_fake_performance_repo")
(repo / "todo").mkdir(parents=True, exist_ok=True)
(repo / "templates").mkdir(parents=True, exist_ok=True)
(repo / "README.md").write_text("# Todo API\nJWT todo CRUD API tests.\n", encoding="utf-8")
(repo / "requirements.txt").write_text("django\ndjangorestframework\n", encoding="utf-8")
(repo / "manage.py").write_text("# placeholder\n", encoding="utf-8")
(repo / "todo" / "urls.py").write_text("from django.urls import path\nurlpatterns = []\n", encoding="utf-8")
(repo / "templates" / "login.html").write_text("<form><input name='username'><input type='password'></form>\n", encoding="utf-8")

In [ ]:
def fake_api(target_url, test_case, **kwargs):
    return {
        "id": test_case.get("id", "API_001"),
        "name": test_case.get("name", "api_smoke"),
        "method": "GET",
        "endpoint": test_case.get("endpoint", "/api/todos/"),
        "status": "passed",
        "expected_status": 200,
        "actual_status": 200,
        "duration_ms": 5.0,
        "details": "mocked API execution",
        "evidence": {},
        "assertions": [{"type": "status_code", "passed": True}],
        "error_type": None,
    }

def fake_ui(target_url, test_case, run_id, discovered_ui_flows=None, user_preferences=None):
    return {
        "id": test_case.get("id", "UI_001"),
        "name": test_case.get("name", "login_page_visible"),
        "flow": "login",
        "status": "passed",
        "target_path": "/login/",
        "target_url": target_url.rstrip("/") + "/login/",
        "duration_ms": 5.0,
        "details": "mocked UI execution",
        "screenshot": None,
        "assertions": [{"type": "login_form_present", "passed": True}],
        "error_type": None,
        "evidence": {},
    }

def fake_perf(target_url, test_case, run_id, user_preferences=None):
    return {
        "id": test_case.get("id", "PERF_001"),
        "name": test_case.get("name", "todo_list_perf"),
        "endpoint": test_case.get("endpoint", "/api/todos/"),
        "method": "GET",
        "status": "performance_threshold_failed",
        "users": 1,
        "spawn_rate": 1.0,
        "duration_seconds": 1,
        "total_requests": 10,
        "failures": 0,
        "failure_rate": 0.0,
        "average_response_time_ms": 2500.0,
        "min_response_time_ms": 100.0,
        "max_response_time_ms": 6000.0,
        "p50_response_time_ms": 2000.0,
        "p95_response_time_ms": 5500.0,
        "requests_per_second": 2.0,
        "threshold_results": [{"name": "p95_response_time", "passed": False, "actual": 5500, "threshold": 5000}],
        "details": "mocked performance threshold failure",
        "error_type": "performance_threshold_failed",
        "artifact_paths": [f"results/runs/{run_id}/performance/locustfile_PERF_001.py"],
    }

api_testing_agent.execute_api_test_case = fake_api
ui_testing_agent.execute_ui_test_case = fake_ui
performance_testing_agent.execute_performance_test_case = fake_perf

In [ ]:
final_state = run_workflow({
    "repo_path": str(repo),
    "target_url": "http://localhost:8000",
    "user_preferences": {
        "test_types": ["api", "ui", "performance"],
        "execution_mode": "sequential",
        "focus": "JWT authentication todo CRUD API tests",
        "planner_use_llm": False,
    },
    "errors": [],
    "agent_logs": [],
})

final_state["performance_results"]["summary"]

In [ ]:
final_state["bug_results"]["summary"], final_state["report_html_path"]

## Part B - Optional real target app

This requires the Django target app running at `http://localhost:8000` and Locust installed. Do not enable this against external sites.

In [ ]:
RUN_REAL_PERFORMANCE_WORKFLOW = False

if RUN_REAL_PERFORMANCE_WORKFLOW:
    real_state = run_workflow({
        "repo_path": str(repo),
        "target_url": "http://localhost:8000",
        "user_preferences": {
            "test_types": ["api", "ui", "performance"],
            "execution_mode": "sequential",
            "focus": "JWT authentication todo CRUD API tests",
            "planner_use_llm": False,
        },
        "errors": [],
        "agent_logs": [],
    })
    print(real_state.get("performance_result_path"))
else:
    print("Skipped real workflow run. Start the local target app first, then set RUN_REAL_PERFORMANCE_WORKFLOW = True.")

## Part C - Graph visualization

In [ ]:
from test_auto.graph.workflow import build_graph

graph = build_graph().get_graph()
try:
    graph.draw_mermaid_png()
    print("Graph image rendered.")
except Exception:
    print(graph.draw_mermaid())